# Transfer Learning with TimesFM-1.0-200m Checkpoint

This notebook demonstrates how to use the pre-trained timesfm-1.0-200m checkpoint for transfer learning on your custom time series datasets.

## What is Transfer Learning in TimesFM?

Transfer learning involves taking a pre-trained TimesFM model and adapting it to your specific domain or dataset. The timesfm-1.0-200m checkpoint provides an excellent starting point with:

- **200M parameters** pre-trained on diverse time series data
- **Context length** up to 512 timepoints
- **Universal forecasting** capabilities across different domains
- **Both PyTorch and JAX** implementations available

## When to Use Transfer Learning?

Transfer learning with timesfm-1.0-200m is particularly beneficial when:

1. **Limited training data**: You have a smaller dataset but want to leverage large-scale pre-training
2. **Domain-specific patterns**: Your data has specific characteristics not fully captured by the general model
3. **Improved accuracy**: You want to fine-tune for better performance on your specific use case
4. **Faster convergence**: Starting from pre-trained weights reduces training time

## Transfer Learning Approaches

1. **Feature extraction**: Use pre-trained model as fixed feature extractor
2. **Fine-tuning**: Update all or some layers of the pre-trained model
3. **Gradual unfreezing**: Start with frozen layers and gradually unfreeze during training


## Setup and Dependencies

First, let's import the necessary libraries and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from typing import Optional, Tuple, List
import matplotlib.pyplot as plt

# TimesFM imports
import timesfm
from timesfm import TimesFm, TimesFmCheckpoint, TimesFmHparams
from finetuning.finetuning_torch import FinetuningConfig, TimesFMFinetuner
from torch.utils.data import Dataset, DataLoader

# For PyTorch transfer learning
from timesfm.pytorch_patched_decoder import PatchedTimeSeriesDecoder, TimesFMConfig

## Method 1: Loading TimesFM-1.0-200m for Transfer Learning

### PyTorch Version

In [ ]:
def load_timesfm_1_0_200m_pytorch(context_len: int = 512, horizon_len: int = 96, device: str = "auto") -> TimesFm:
    """
    Load the timesfm-1.0-200m PyTorch checkpoint for transfer learning.
    
    Args:
        context_len: Maximum context length (must be multiple of 32, max 512 for 1.0 model)
        horizon_len: Forecast horizon length
        device: Device to run on ("auto", "cpu", or "gpu")
    
    Returns:
        TimesFM model instance ready for transfer learning
    """
    # Determine backend
    if device == "auto":
        backend = "gpu" if torch.cuda.is_available() else "cpu"
    else:
        backend = device
    
    # Validate context length for 1.0 model
    if context_len > 512:
        raise ValueError("timesfm-1.0-200m supports maximum context length of 512")
    if context_len % 32 != 0:
        raise ValueError("context_len must be a multiple of 32")
    
    # Initialize model with 1.0-200m checkpoint
    tfm = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(
            backend=backend,
            per_core_batch_size=32,
            horizon_len=horizon_len,
            context_len=context_len,
        ),
        checkpoint=timesfm.TimesFmCheckpoint(
            huggingface_repo_id="google/timesfm-1.0-200m-pytorch"
        ),
    )
    
    print(f"Loaded timesfm-1.0-200m-pytorch with:")
    print(f"  - Context length: {context_len}")
    print(f"  - Horizon length: {horizon_len}")
    print(f"  - Backend: {backend}")
    
    return tfm

# Load the model
model = load_timesfm_1_0_200m_pytorch(context_len=256, horizon_len=96)

### JAX Version

In [ ]:
def load_timesfm_1_0_200m_jax(context_len: int = 512, horizon_len: int = 96, device: str = "auto") -> TimesFm:
    """
    Load the timesfm-1.0-200m JAX checkpoint for transfer learning.
    
    Args:
        context_len: Maximum context length (must be multiple of 32, max 512 for 1.0 model)
        horizon_len: Forecast horizon length
        device: Device to run on ("auto", "cpu", or "gpu")
    
    Returns:
        TimesFM model instance ready for transfer learning
    """
    # Determine backend
    if device == "auto":
        backend = "gpu" if torch.cuda.is_available() else "cpu"
    else:
        backend = device
    
    # Validate context length for 1.0 model
    if context_len > 512:
        raise ValueError("timesfm-1.0-200m supports maximum context length of 512")
    if context_len % 32 != 0:
        raise ValueError("context_len must be a multiple of 32")
    
    # Initialize model with 1.0-200m checkpoint
    tfm = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(
            backend=backend,
            per_core_batch_size=32,
            horizon_len=horizon_len,
            context_len=context_len,
        ),
        checkpoint=timesfm.TimesFmCheckpoint(
            huggingface_repo_id="google/timesfm-1.0-200m"
        ),
    )
    
    print(f"Loaded timesfm-1.0-200m with:")
    print(f"  - Context length: {context_len}")
    print(f"  - Horizon length: {horizon_len}")
    print(f"  - Backend: {backend}")
    
    return tfm

# Uncomment to use JAX version instead
# model = load_timesfm_1_0_200m_jax(context_len=256, horizon_len=96)

## Method 2: Transfer Learning with Fine-tuning

For domain-specific adaptation, you can fine-tune the pre-trained timesfm-1.0-200m model on your data.

In [ ]:
class TransferLearningDataset(Dataset):
    """
    Dataset class for transfer learning with TimesFM-1.0-200m.
    
    This class prepares your time series data for transfer learning by:
    - Creating appropriate context-horizon splits
    - Handling different frequency types
    - Normalizing data if needed
    """
    
    def __init__(self, 
                 time_series: List[np.ndarray], 
                 context_length: int, 
                 horizon_length: int,
                 freq_type: int = 0,
                 normalize: bool = True):
        """
        Initialize the transfer learning dataset.
        
        Args:
            time_series: List of time series arrays
            context_length: Number of past timesteps to use
            horizon_length: Number of future timesteps to predict
            freq_type: Frequency type (0=high freq, 1=medium freq, 2=low freq)
            normalize: Whether to normalize the data
        """
        self.time_series = time_series
        self.context_length = context_length
        self.horizon_length = horizon_length
        self.freq_type = freq_type
        self.normalize = normalize
        
        self._prepare_samples()
    
    def _prepare_samples(self):
        """Prepare sliding window samples for transfer learning."""
        self.samples = []
        total_length = self.context_length + self.horizon_length
        
        for ts in self.time_series:
            if len(ts) < total_length:
                continue
                
            # Optional normalization for transfer learning
            if self.normalize:
                ts_mean = np.mean(ts)
                ts_std = np.std(ts) + 1e-8
                ts = (ts - ts_mean) / ts_std
            
            # Create sliding windows
            for start_idx in range(0, len(ts) - total_length + 1, self.horizon_length):
                end_idx = start_idx + self.context_length
                context = ts[start_idx:end_idx]
                horizon = ts[end_idx:end_idx + self.horizon_length]
                self.samples.append((context, horizon))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        context, horizon = self.samples[idx]
        
        context_tensor = torch.tensor(context, dtype=torch.float32)
        horizon_tensor = torch.tensor(horizon, dtype=torch.float32)
        freq_tensor = torch.tensor(self.freq_type, dtype=torch.long)
        
        return context_tensor, horizon_tensor, freq_tensor, freq_tensor

### Transfer Learning Configuration

In [ ]:
def create_transfer_learning_config(learning_rate: float = 1e-4,
                                   num_epochs: int = 10,
                                   batch_size: int = 16,
                                   freeze_layers: Optional[List[str]] = None) -> FinetuningConfig:
    """
    Create configuration for transfer learning with timesfm-1.0-200m.
    
    Args:
        learning_rate: Learning rate for transfer learning (typically lower than training from scratch)
        num_epochs: Number of epochs for fine-tuning
        batch_size: Batch size for training
        freeze_layers: List of layer names to freeze during transfer learning
    
    Returns:
        FinetuningConfig for transfer learning
    """
    config = FinetuningConfig(
        learning_rate=learning_rate,
        num_epochs=num_epochs,
        batch_size=batch_size,
        weight_decay=1e-4,  # Regularization for transfer learning
        warmup_steps=100,   # Gentle warmup for pre-trained model
        save_every_n_epochs=2,
        eval_every_n_epochs=1,
        patience=5,         # Early stopping patience
        min_delta=1e-6,     # Minimum improvement threshold
    )
    
    return config

# Example configuration for different transfer learning scenarios

# Conservative transfer learning (minimal changes to pre-trained model)
conservative_config = create_transfer_learning_config(
    learning_rate=1e-5,
    num_epochs=5,
    batch_size=8
)

# Standard transfer learning
standard_config = create_transfer_learning_config(
    learning_rate=1e-4,
    num_epochs=10,
    batch_size=16
)

# Aggressive transfer learning (more adaptation)
aggressive_config = create_transfer_learning_config(
    learning_rate=1e-3,
    num_epochs=20,
    batch_size=32
)

## Method 3: Feature Extraction with Frozen Model

Use the pre-trained timesfm-1.0-200m as a feature extractor without fine-tuning.

In [ ]:
def extract_features_timesfm_200m(model: TimesFm, 
                                 time_series: List[np.ndarray],
                                 freq_type: int = 0) -> np.ndarray:
    """
    Extract features from time series using pre-trained timesfm-1.0-200m.
    
    This approach treats the model as a fixed feature extractor,
    which is useful when you want to:
    - Preserve the pre-trained representations
    - Use features for downstream tasks
    - Avoid overfitting on small datasets
    
    Args:
        model: Pre-trained TimesFM model
        time_series: List of time series to extract features from
        freq_type: Frequency type for the time series
    
    Returns:
        Extracted features as numpy array
    """
    features = []
    
    for ts in time_series:
        # Use the model's internal representations (this is a simplified example)
        # In practice, you might need to access intermediate layers
        try:
            # Generate forecast to get model's internal processing
            forecast, _ = model.forecast(
                [ts],
                freq=[freq_type]
            )
            # Use forecast as feature representation
            features.append(forecast[0])
        except Exception as e:
            print(f"Error processing time series: {e}")
            continue
    
    return np.array(features)

# Example usage
def demonstrate_feature_extraction():
    """
    Demonstrate feature extraction with timesfm-1.0-200m.
    """
    # Generate sample time series
    sample_ts = [
        np.sin(np.linspace(0, 10, 200)) + np.random.normal(0, 0.1, 200),
        np.cos(np.linspace(0, 8, 200)) + np.random.normal(0, 0.1, 200),
    ]
    
    # Extract features
    print("Extracting features using timesfm-1.0-200m...")
    features = extract_features_timesfm_200m(model, sample_ts, freq_type=0)
    
    print(f"Extracted features shape: {features.shape}")
    return features

# Uncomment to run feature extraction
# features = demonstrate_feature_extraction()

## Best Practices for Transfer Learning with TimesFM-1.0-200m

### 1. Data Preparation

In [ ]:
def prepare_data_for_transfer_learning(time_series_data: List[np.ndarray],
                                     target_context_len: int = 256,
                                     target_horizon_len: int = 96) -> Tuple[List[np.ndarray], List[np.ndarray]]:
    """
    Prepare time series data for transfer learning with timesfm-1.0-200m.
    
    Best practices:
    - Ensure sufficient history for context
    - Handle missing values appropriately
    - Consider data normalization
    - Split data appropriately for validation
    
    Args:
        time_series_data: Raw time series data
        target_context_len: Desired context length
        target_horizon_len: Desired horizon length
    
    Returns:
        Tuple of (training_data, validation_data)
    """
    processed_data = []
    min_length = target_context_len + target_horizon_len
    
    for ts in time_series_data:
        # Skip series that are too short
        if len(ts) < min_length:
            continue
            
        # Handle missing values
        if np.any(np.isnan(ts)):
            # Simple forward fill (you may want more sophisticated imputation)
            ts = pd.Series(ts).fillna(method='ffill').fillna(method='bfill').values
        
        # Optional: Remove outliers (simple approach)
        q1, q3 = np.percentile(ts, [25, 75])
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        ts = np.clip(ts, lower_bound, upper_bound)
        
        processed_data.append(ts)
    
    # Split into train/validation (80/20 split)
    split_idx = int(0.8 * len(processed_data))
    train_data = processed_data[:split_idx]
    val_data = processed_data[split_idx:]
    
    print(f"Prepared {len(train_data)} training series and {len(val_data)} validation series")
    print(f"Context length: {target_context_len}, Horizon length: {target_horizon_len}")
    
    return train_data, val_data

### 2. Transfer Learning Strategies

In [ ]:
def choose_transfer_learning_strategy(dataset_size: int, 
                                    domain_similarity: str = "similar") -> dict:
    """
    Choose the best transfer learning strategy based on dataset characteristics.
    
    Args:
        dataset_size: Number of time series in your dataset
        domain_similarity: How similar your domain is to pre-training data
                          ("very_similar", "similar", "different", "very_different")
    
    Returns:
        Dictionary with recommended strategy and parameters
    """
    strategies = {
        "very_small": {  # < 100 series
            "very_similar": {
                "approach": "feature_extraction",
                "learning_rate": 0,
                "freeze_layers": "all",
                "description": "Use pre-trained model as feature extractor only"
            },
            "similar": {
                "approach": "fine_tune_head",
                "learning_rate": 1e-5,
                "freeze_layers": "backbone",
                "description": "Fine-tune only the final layers"
            },
            "different": {
                "approach": "light_fine_tuning",
                "learning_rate": 1e-4,
                "freeze_layers": "early_layers",
                "description": "Fine-tune later layers with low learning rate"
            }
        },
        "small": {  # 100-1000 series
            "very_similar": {
                "approach": "fine_tune_head",
                "learning_rate": 1e-4,
                "freeze_layers": "backbone",
                "description": "Fine-tune head with moderate learning rate"
            },
            "similar": {
                "approach": "gradual_unfreezing",
                "learning_rate": 1e-4,
                "freeze_layers": "gradual",
                "description": "Gradually unfreeze layers during training"
            },
            "different": {
                "approach": "full_fine_tuning",
                "learning_rate": 5e-4,
                "freeze_layers": "none",
                "description": "Fine-tune all layers with careful regularization"
            }
        },
        "large": {  # > 1000 series
            "very_similar": {
                "approach": "gradual_unfreezing",
                "learning_rate": 1e-4,
                "freeze_layers": "gradual",
                "description": "Gradually unfreeze with standard learning rate"
            },
            "similar": {
                "approach": "full_fine_tuning",
                "learning_rate": 1e-3,
                "freeze_layers": "none",
                "description": "Full fine-tuning with higher learning rate"
            },
            "different": {
                "approach": "full_fine_tuning",
                "learning_rate": 1e-3,
                "freeze_layers": "none",
                "description": "Aggressive fine-tuning for domain adaptation"
            }
        }
    }
    
    # Determine dataset size category
    if dataset_size < 100:
        size_category = "very_small"
    elif dataset_size < 1000:
        size_category = "small"
    else:
        size_category = "large"
    
    strategy = strategies[size_category][domain_similarity]
    strategy["dataset_size"] = dataset_size
    strategy["size_category"] = size_category
    
    return strategy

# Example usage
print("Transfer Learning Strategy Recommendations:")
print("="*50)

# Small dataset, similar domain
strategy1 = choose_transfer_learning_strategy(50, "similar")
print(f"Small dataset (50 series), similar domain:")
print(f"  Approach: {strategy1['approach']}")
print(f"  Learning rate: {strategy1['learning_rate']}")
print(f"  Description: {strategy1['description']}")
print()

# Large dataset, different domain
strategy2 = choose_transfer_learning_strategy(5000, "different")
print(f"Large dataset (5000 series), different domain:")
print(f"  Approach: {strategy2['approach']}")
print(f"  Learning rate: {strategy2['learning_rate']}")
print(f"  Description: {strategy2['description']}")

## Complete Transfer Learning Example

Here's a complete example showing how to use timesfm-1.0-200m for transfer learning:

In [ ]:
def complete_transfer_learning_example():
    """
    Complete example of transfer learning with timesfm-1.0-200m.
    """
    print("TimesFM-1.0-200m Transfer Learning Example")
    print("="*50)
    
    # Step 1: Generate synthetic data (replace with your data)
    print("Step 1: Preparing data...")
    np.random.seed(42)
    synthetic_data = []
    for i in range(20):
        # Create diverse synthetic time series
        t = np.linspace(0, 4*np.pi, 400)
        ts = (np.sin(t + i*0.5) + 0.5*np.sin(3*t + i) + 
              0.1*np.random.normal(0, 1, len(t)) + i*0.1)
        synthetic_data.append(ts)
    
    # Step 2: Choose strategy
    print("Step 2: Choosing transfer learning strategy...")
    strategy = choose_transfer_learning_strategy(
        dataset_size=len(synthetic_data),
        domain_similarity="similar"
    )
    print(f"Recommended approach: {strategy['approach']}")
    print(f"Learning rate: {strategy['learning_rate']}")
    
    # Step 3: Prepare data
    print("Step 3: Preparing data for transfer learning...")
    train_data, val_data = prepare_data_for_transfer_learning(
        synthetic_data, 
        target_context_len=256, 
        target_horizon_len=64
    )
    
    # Step 4: Load pre-trained model
    print("Step 4: Loading timesfm-1.0-200m checkpoint...")
    try:
        model = load_timesfm_1_0_200m_pytorch(
            context_len=256, 
            horizon_len=64
        )
        print("Model loaded successfully!")
    except Exception as e:
        print(f"Note: Model loading requires proper installation. Error: {e}")
        return
    
    # Step 5: Create dataset
    print("Step 5: Creating transfer learning dataset...")
    dataset = TransferLearningDataset(
        time_series=train_data,
        context_length=256,
        horizon_length=64,
        freq_type=0,  # High frequency data
        normalize=True
    )
    print(f"Created dataset with {len(dataset)} samples")
    
    # Step 6: Set up transfer learning configuration
    print("Step 6: Configuring transfer learning...")
    config = create_transfer_learning_config(
        learning_rate=strategy['learning_rate'],
        num_epochs=5,  # Quick example
        batch_size=8
    )
    
    print("Transfer learning setup complete!")
    print("\nNext steps:")
    print("1. Run fine-tuning with TimesFMFinetuner")
    print("2. Evaluate on validation data")
    print("3. Deploy for inference")
    
    return model, dataset, config

# Run the complete example
try:
    result = complete_transfer_learning_example()
    if result:
        model, dataset, config = result
except Exception as e:
    print(f"Example requires full TimesFM installation: {e}")

## Evaluation and Validation

After transfer learning, it's important to properly evaluate your model:

In [ ]:
def evaluate_transfer_learning_model(model: TimesFm, 
                                   test_data: List[np.ndarray],
                                   context_len: int = 256,
                                   horizon_len: int = 64) -> dict:
    """
    Evaluate the transfer learning model performance.
    
    Args:
        model: Fine-tuned TimesFM model
        test_data: Test time series data
        context_len: Context length used for evaluation
        horizon_len: Horizon length for evaluation
    
    Returns:
        Dictionary with evaluation metrics
    """
    predictions = []
    ground_truth = []
    
    for ts in test_data:
        if len(ts) < context_len + horizon_len:
            continue
            
        # Split into context and target
        context = ts[:context_len]
        target = ts[context_len:context_len + horizon_len]
        
        try:
            # Generate forecast
            forecast, _ = model.forecast([context], freq=[0])
            predictions.append(forecast[0][:horizon_len])
            ground_truth.append(target)
        except Exception as e:
            print(f"Error in forecasting: {e}")
            continue
    
    if not predictions:
        return {"error": "No successful predictions"}
    
    # Calculate metrics
    predictions = np.array(predictions)
    ground_truth = np.array(ground_truth)
    
    mae = np.mean(np.abs(predictions - ground_truth))
    mse = np.mean((predictions - ground_truth)**2)
    rmse = np.sqrt(mse)
    
    # MAPE (handling zero values)
    non_zero_mask = ground_truth != 0
    if np.any(non_zero_mask):
        mape = np.mean(np.abs((ground_truth[non_zero_mask] - predictions[non_zero_mask]) / ground_truth[non_zero_mask])) * 100
    else:
        mape = float('inf')
    
    metrics = {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape,
        "num_predictions": len(predictions)
    }
    
    return metrics

def plot_transfer_learning_results(predictions: np.ndarray, 
                                  ground_truth: np.ndarray,
                                  context: np.ndarray,
                                  title: str = "Transfer Learning Results"):
    """
    Plot the results of transfer learning.
    """
    plt.figure(figsize=(12, 6))
    
    # Plot context
    context_x = range(len(context))
    plt.plot(context_x, context, 'b-', label='Context', linewidth=2)
    
    # Plot predictions and ground truth
    horizon_x = range(len(context), len(context) + len(predictions))
    plt.plot(horizon_x, predictions, 'r--', label='Prediction', linewidth=2)
    plt.plot(horizon_x, ground_truth, 'g-', label='Ground Truth', linewidth=2)
    
    plt.axvline(x=len(context), color='black', linestyle=':', alpha=0.7, label='Forecast Start')
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print("Evaluation functions defined. Use after model training to assess performance.")

## Summary and Next Steps

This notebook demonstrated how to use the timesfm-1.0-200m checkpoint for transfer learning:

### Key Takeaways:

1. **Model Loading**: Use the appropriate checkpoint (PyTorch or JAX) for your setup
2. **Strategy Selection**: Choose the right approach based on dataset size and domain similarity
3. **Data Preparation**: Properly format and preprocess your time series data
4. **Configuration**: Set appropriate hyperparameters for transfer learning
5. **Evaluation**: Validate your results with proper metrics

### Best Practices:

- Start with conservative learning rates (1e-5 to 1e-4)
- Use early stopping to prevent overfitting
- Consider gradual unfreezing for large datasets
- Always validate on held-out data
- Monitor training closely for signs of overfitting

### Next Steps:

1. **Load your own data**: Replace synthetic examples with your time series
2. **Run full training**: Use the TimesFMFinetuner for complete fine-tuning
3. **Hyperparameter tuning**: Experiment with different configurations
4. **Model deployment**: Deploy your fine-tuned model for production use
5. **Continuous improvement**: Monitor performance and retrain as needed

For more examples and detailed documentation, see:
- [Finetuning Notebook](./finetuning_torch.ipynb)
- [TimesFM Repository](https://github.com/google-research/timesfm)
- [Hugging Face Models](https://huggingface.co/collections/google/timesfm-release-66e4be5fdb56e960c1e482a6)